# 0) Gruplanmis Dataset Hazirlama (Colab)

Notebook 2 egitiminden once kopya farkindalikli, source-aware ve taksonomi uyumlu dataset hazirlama akisidir.

Repo icinde hazir class-root datasetler: data/class_root_dataset/tomato_fruit, data/class_root_dataset/grape_fruit ve data/class_root_dataset/grape_leaf.

Audit hedefi sadece "temiz" veri degil, adapter performansi icin guvenilir veri uretmek olmalidir:
- exact duplicate ve near-duplicate ailelerini birlikte tutun.
- ayni capture/session/source ailelerini ayirmayin; source leakage'i engelleyin.
- gercek hard negatives ve OOD pool'u klasification siniflarindan ayri yonetin.
- sadece audit raporu temiz oldugu icin degil, class support ve source dagilimi guvenilir oldugu icin materyalize edin.

Onerilen kullanim sirasi:
1. Parametreleri duzenleyin.
2. Guncelleme/erisim kontrolu hucresini calistirin.
3. Audit hucresini calistirin.
4. Cikti dosyalarini `guided/00_start_here.md` ile okuyun.
5. Her sey temizse runtime dataset materyalizasyonunu acin.


## Repo-Ici Dataset Secimi

Notebook 0, repo icindeki class-root datasetleri audit edip runtime dataset'e cevirir. Asagidaki adimlari izleyin:

### Nasil Kullanilir

1. `REPO_DATASET_ROOT` ile datasetlerin bulundugu repo klasorunu secin.
2. `REPO_DATASET_NAME` ile alt dataset adini belirtin ya da bos birakip notebook'un sormasini bekleyin.
3. `DATASET_ROOT` bos ise yukaridaki secim kullanilir.
4. Audit ve materyalizasyon hucrelerini sirayla calistirin.

### Ornek

```python
REPO_DATASET_ROOT = "data/class_root_dataset"
REPO_DATASET_NAME = "tomato_fruit"
```

Bu akiste veri importu Drive'a bagimli degildir; Notebook 0 repo icindeki veri kaynaklarini audit edip runtime dataset'e cevirir.

In [ ]:
# Bootstrap: use helper to keep notebook tidy
from scripts.colab_repo_bootstrap import probe_repo_update_status
from scripts.colab_repo_bootstrap import _ensure_repo_root_for_update_check
repo_root_for_update_check = _ensure_repo_root_for_update_check()

def _build_repo_access_url(base: str, params: dict) -> str:
    """Build repo access URL for freshness check (validation contract)."""
    from urllib.parse import urlencode
    return f"{base}?{urlencode(params)}"

# [KONTROL] Ilk hucre: Bootstrap kontrati
from scripts.notebook_helpers.nb0_grouped_dataset_prep_helpers import run_bootstrap_notebook
BOOTSTRAP = run_bootstrap_notebook(notebook_name="Notebook 0: Dataset Preparation", require_colab_requirements=True, auto_clone_repo=True)
ROOT = BOOTSTRAP["ROOT"]
print('Bootstrap complete')


In [ ]:
# Init environment and telemetry via helper to reduce noise in the notebook
from scripts.notebook_helpers.nb0_grouped_dataset_prep_helpers import init_run_environment
env = init_run_environment(ROOT)
STATE = env['STATE']
RUN_ID = env['RUN_ID']
NOTEBOOK_FILENAME = env['NOTEBOOK_FILENAME']
REPO_RUN_DIR = env['REPO_RUN_DIR']
REPO_NOTEBOOK_OUTPUT_PATH = env['REPO_NOTEBOOK_OUTPUT_PATH']
LOCAL_OUTPUT_DIR = env['LOCAL_OUTPUT_DIR']
REPO_OUTPUT_DIR = env['REPO_OUTPUT_DIR']
REPO_TELEMETRY_DIR = env['REPO_TELEMETRY_DIR']
CHECKPOINT_ROOT = env['CHECKPOINT_ROOT']
TELEMETRY = env['TELEMETRY']
DEVICE = env['DEVICE']
EMBEDDING_BATCH_SIZE = env['EMBEDDING_BATCH_SIZE']
NEIGHBORS = env['NEIGHBORS']
print(f'[BOOTSTRAP] run_id={RUN_ID}')

In [ ]:
with TELEMETRY.capture_cell_output("Cell 3: Parameters"):
    # --- Notebook 0 parametreleri ---
    # Once bu hucreyi duzenleyin, sonra alttaki hucreleri sirayla calistirin.

    # --- HIZLI KULLANIM: Genelde sadece bu satirlari degistirin ---
    # REPO_DATASET_ROOT: repo ici relatif yol veya mutlak yol olabilir.
    REPO_DATASET_ROOT = "data/class_root_dataset"

    # REPO_DATASET_NAME: parent klasor verdiyseniz dataset adi; dogrudan dataset yolu verdiyseniz bos birakin.
    REPO_DATASET_NAME = ""

    # DATASET_ROOT: opsiyonel override. Bos ise yukaridaki secim kullanilir.
    DATASET_ROOT = ""

    # CROP_NAME / PART_NAME: bos birakirsan notebook sorar.
    CROP_NAME = ""
    PART_NAME = ""

    # AUTORUN_AUDIT: True ise audit hucreleri varsayilan akista calisir.
    AUTORUN_AUDIT = True

    # AUTORUN_MATERIALIZE: True ise runtime dataset materyalizasyonu etkin olur.
    AUTORUN_MATERIALIZE = False

    # Optional Drive import controls remain available for manual workflows.
    IMPORT_FROM_DRIVE = True
    DRIVE_DATASET_PATH = ""
    DRIVE_DATASET_NAME = ""
    
    # Dataset preparation parameters
    PREP_ARTIFACT_ROOT = "outputs/ood_prep_reports"
    PREPARED_CLASS_ROOT = "data/prepared_class_root_datasets"
    PREPARED_RUNTIME_ROOT = "data/prepared_runtime_datasets"
    
    # Cleanup and validation
    CLEANUP_SEED = 42
    PREPARE_DATASET_FROM_REPORTS = True
    MATERIALIZE_AFTER_REVIEW = False
    
    # OOD dataset configuration
    OOD_ROOT = ""
    OOD_DATASET_NAME = ""
    OOD_DATASET_ROOT = "data/ood_dataset"
    ASK_FOR_OOD_ROOT = False
    
    # GitHub push settings
    SAVE_RUNTIME_DATASET_TO_GITHUB = False
    RUNTIME_DATASET_PUSH_REMOTE_NAME = "origin"
    RUNTIME_DATASET_PUSH_BRANCH = "main"
    
    # Under-min-eval policy
    UNDER_MIN_EVAL_POLICY = {}


In [ ]:
with TELEMETRY.capture_cell_output(\"Cell 3a: Google Drive Dataset Import (Opsiyonel)\"):\n
    from scripts.colab_dataset_layout import resolve_dataset_directory_from_parent\n
    from scripts.colab_repo_bootstrap import mount_drive_if_available\n
    import shutil\n
    from scripts.notebook_helpers.nb0_grouped_dataset_prep_helpers import import_dataset_from_drive, _drive_destination_parent\n
    # Google Drive'dan dataset alma secenegi (parametrelerden al)\n
    if IMPORT_FROM_DRIVE and str(DRIVE_DATASET_PATH).strip():\n
        print(\"[DRIVE] Google Drive icinde dataset ara...\")\n
        mount_drive_if_available()\n
        drive_path = Path(DRIVE_DATASET_PATH).expanduser()\n
        if not drive_path.exists():\n
            raise RuntimeError(f\"Drive yolu bulunamadi: {drive_path}\")\n
        dataset_name, source, available_datasets = resolve_dataset_directory_from_parent(\n
            dataset_parent=drive_path,\n
            requested_name=DRIVE_DATASET_NAME,\n
            prompt_label=\"Drive dataset\",\n
        )\n
        print(f\"[DRIVE] secilebilir datasetler={available_datasets}\")\n
        dest_parent = _drive_destination_parent(DATASET_ROOT, ROOT)\n
        if import_dataset_from_drive(source, dest_parent, dataset_name):\n
            DATASET_ROOT = str(dest_parent / dataset_name)\n
            print(f\"[DRIVE] DATASET_ROOT guncellendi: {DATASET_ROOT}\")\n
        else:\n
            existing_target = dest_parent / dataset_name\n
            if existing_target.is_dir():\n
                DATASET_ROOT = str(existing_target)\n
                print(f\"[DRIVE] Mevcut kopya kullaniliyor, DATASET_ROOT guncellendi: {DATASET_ROOT}\")\n
    else:\n
        print(\"[DRIVE] IMPORT_FROM_DRIVE=False veya DRIVE_DATASET_PATH bos. Drive import atlanacak.\")\n

In [ ]:
# Access and update check
with TELEMETRY.capture_cell_output("Cell 3b: Access and Update Check"):
    from scripts.colab_repo_bootstrap import collect_notebook_access_report, print_notebook_access_report

    PREP_DINOV3_MODEL_ID = 'facebook/dinov3-vitl16-pretrain-lvd1689m'
    PREP_BIOCLIP_MODEL_ID = 'imageomics/bioclip-2.5-vith14'

    access_report = collect_notebook_access_report(
        repo_root=ROOT,
        hf_model_ids=[PREP_DINOV3_MODEL_ID, PREP_BIOCLIP_MODEL_ID],
    )
    STATE["access_report"] = access_report
    print_notebook_access_report(access_report, print_fn=print)
    TELEMETRY.update_latest(
        {
            "phase": "access_checked",
            "github_read_access": access_report.get("github", {}).get("read_access_mode"),
            "repo_update_relation": access_report.get("repo_updates", {}).get("relation"),
            "hf_access_mode": access_report.get("huggingface", {}).get("access_mode"),
        }
    )

In [ ]:
with TELEMETRY.capture_cell_output("Cell 4: Dataset Audit"):
    from scripts.notebook_helpers.nb0_grouped_dataset_prep_helpers import run_dataset_audit
    STATE, CROP_NAME, PART_NAME = run_dataset_audit(
        ROOT=ROOT,
        STATE=STATE,
        TELEMETRY=TELEMETRY,
        DEVICE=DEVICE,
        EMBEDDING_BATCH_SIZE=EMBEDDING_BATCH_SIZE,
        NEIGHBORS=NEIGHBORS,
        REPO_DATASET_ROOT=REPO_DATASET_ROOT,
        REPO_DATASET_NAME=REPO_DATASET_NAME,
        DATASET_ROOT=DATASET_ROOT,
        CROP_NAME=CROP_NAME,
        PART_NAME=PART_NAME,
        PREP_ARTIFACT_ROOT=PREP_ARTIFACT_ROOT,
        PREP_DINOV3_MODEL_ID=PREP_DINOV3_MODEL_ID,
        PREP_BIOCLIP_MODEL_ID=PREP_BIOCLIP_MODEL_ID,
        UNDER_MIN_EVAL_POLICY=UNDER_MIN_EVAL_POLICY,
    )


In [ ]:
with TELEMETRY.capture_cell_output("Cell 4b: Prepare Dataset For Materialization"):
    from scripts.notebook_helpers.nb0_grouped_dataset_prep_helpers import run_prepare_dataset
    run_prepare_dataset(
        ROOT=ROOT,
        STATE=STATE,
        TELEMETRY=TELEMETRY,
        DEVICE=DEVICE,
        EMBEDDING_BATCH_SIZE=EMBEDDING_BATCH_SIZE,
        NEIGHBORS=NEIGHBORS,
        CROP_NAME=CROP_NAME,
        PART_NAME=PART_NAME,
        PREPARED_CLASS_ROOT=PREPARED_CLASS_ROOT,
        CLEANUP_SEED=CLEANUP_SEED,
        PREP_DINOV3_MODEL_ID=PREP_DINOV3_MODEL_ID,
        PREP_BIOCLIP_MODEL_ID=PREP_BIOCLIP_MODEL_ID,
        UNDER_MIN_EVAL_POLICY=UNDER_MIN_EVAL_POLICY,
        PREPARE_DATASET_FROM_REPORTS=PREPARE_DATASET_FROM_REPORTS,
    )


In [ ]:
# Fix .gitignore for prepared_runtime_datasets via helper to declutter notebook
from scripts.notebook_helpers.nb0_grouped_dataset_prep_helpers import fix_gitignore
fix_gitignore(ROOT)

In [ ]:
with TELEMETRY.capture_cell_output("Cell 5: Materialize Runtime Dataset"):
    from scripts.notebook_helpers.nb0_grouped_dataset_prep_helpers import run_materialize_runtime_dataset, save_run_outputs_to_repo
    run_materialize_runtime_dataset(
        ROOT=ROOT,
        STATE=STATE,
        TELEMETRY=TELEMETRY,
        CROP_NAME=CROP_NAME,
        PART_NAME=PART_NAME,
        OOD_ROOT=OOD_ROOT,
        OOD_DATASET_NAME=OOD_DATASET_NAME,
        OOD_DATASET_ROOT=OOD_DATASET_ROOT,
        ASK_FOR_OOD_ROOT=ASK_FOR_OOD_ROOT,
        PREPARED_RUNTIME_ROOT=PREPARED_RUNTIME_ROOT,
        MATERIALIZE_AFTER_REVIEW=MATERIALIZE_AFTER_REVIEW,
        SAVE_RUNTIME_DATASET_TO_GITHUB=SAVE_RUNTIME_DATASET_TO_GITHUB,
        RUNTIME_DATASET_PUSH_REMOTE_NAME=RUNTIME_DATASET_PUSH_REMOTE_NAME,
        RUNTIME_DATASET_PUSH_BRANCH=RUNTIME_DATASET_PUSH_BRANCH,
        REPO_NOTEBOOK_OUTPUT_PATH=REPO_NOTEBOOK_OUTPUT_PATH,
        REPO_RUN_DIR=REPO_RUN_DIR,
        REPO_RUN_EXPORTS={},
    )
    
    # Generate final export report
    REPO_RUN_EXPORTS = save_run_outputs_to_repo(LOCAL_OUTPUT_DIR, REPO_OUTPUT_DIR, TELEMETRY, CHECKPOINT_ROOT, REPO_TELEMETRY_DIR, REPO_CHECKPOINT_STATE_DIR)
    for key in sorted(REPO_RUN_EXPORTS):
        print(f"[RUNS] {key} -> {REPO_RUN_EXPORTS[key]}")
    print(f"[RUNS] notebook -> {REPO_NOTEBOOK_OUTPUT_PATH}")
